In [2]:
#!/usr/bin/env python3
# trip_request_producer.py

import json, time, random, uuid, boto3, io, pyarrow.parquet as pq
from kafka import KafkaProducer
from datetime import datetime
import re
from collections import defaultdict
from botocore.config import Config
import urllib3

# Désactiver les warnings SSL
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

# ========== CONFIGURATION ==========
KAFKA_BROKER   = "172.18.0.2:9092"
TOPIC_TRIPS    = "raw.trips"
MINIO_ENDPOINT = "http://100.104.4.32:9000"
MINIO_KEY      = "minioadmin"
MINIO_SECRET   = "minioadmin123"
BUCKET         = "simulated"
PREFIX         = "casa-trajectories/final_89688/"
SPEED_FACTOR   = 10

HOURLY_WEIGHTS = {
    0:0.15,1:0.10,2:0.08,3:0.06,4:0.07,5:0.12,6:0.30,7:0.75,8:1.00,
    9:0.85,10:0.65,11:0.60,12:0.55,13:0.50,14:0.55,15:0.65,16:0.80,
    17:1.00,18:0.95,19:0.75,20:0.55,21:0.45,22:0.35,23:0.25
}
FRIDAY_OVERRIDE = {12:0.20, 13:0.15, 14:0.25}

# ========== CONNEXION MINIO AVEC TIMEOUT AUGMENTÉ ==========
s3 = boto3.client("s3",
    endpoint_url=MINIO_ENDPOINT,
    aws_access_key_id=MINIO_KEY,
    aws_secret_access_key=MINIO_SECRET,
    config=Config(
        signature_version='s3v4',
        read_timeout=120,      # 120 secondes
        connect_timeout=60,    # 60 secondes
        retries={'max_attempts': 3}
    ),
    verify=False
)

def read_parquet_via_boto3_with_retry(bucket, key, max_retries=3):
    """Lit un fichier Parquet avec retry en cas de timeout"""
    for attempt in range(max_retries):
        try:
            print(f"   Tentative {attempt+1}/{max_retries}...")
            response = s3.get_object(Bucket=bucket, Key=key)
            return pq.read_table(io.BytesIO(response['Body'].read()))
        except Exception as e:
            print(f"   ⚠️ Erreur tentative {attempt+1}: {e}")
            if attempt < max_retries - 1:
                time.sleep(5)  # Attendre 5 secondes avant de réessayer
            else:
                raise

def extract_coords_from_polyline(polyline_str):
    try:
        coords = json.loads(polyline_str)
        if coords and len(coords) > 0:
            start_lon, start_lat = coords[0][0], coords[0][1]
            end_lon, end_lat = coords[-1][0], coords[-1][1]
            return start_lon, start_lat, end_lon, end_lat
    except:
        pass
    return None, None, None, None

# Lister les fichiers Parquet
print("📋 Liste des fichiers Parquet...")
response = s3.list_objects_v2(Bucket=BUCKET, Prefix=PREFIX)
parquet_files = [obj["Key"] for obj in response.get("Contents", []) if obj["Key"].endswith(".parquet")]
print(f"📁 {len(parquet_files)} fichiers trouvés")
for f in parquet_files:
    print(f"   - {f}")

# ========== COMPTER LES PAIRES ==========
pair_counts = defaultdict(int)
call_counts = defaultdict(int)
total = 0

for file_key in parquet_files:
    print(f"\n📖 Lecture de {file_key}")
    
    try:
        table = read_parquet_via_boto3_with_retry(BUCKET, file_key)
    except Exception as e:
        print(f"   ❌ Échec lecture {file_key}: {e}")
        continue
    
    cols = table.column_names
    print(f"   Colonnes: {cols}")
    
    if "POLYLINE_CASA" not in cols or "CALL_TYPE" not in cols:
        print(f"   ⚠️ Colonnes manquantes, skip")
        continue
    
    # Extraire origin_zone_id depuis le chemin
    match = re.search(r"origin_zone_id[=:](\d+)", file_key)
    if not match:
        print(f"   ⚠️ origin_zone_id non trouvé, skip")
        continue
    origin = int(match.group(1))
    
    polylines = table.column("POLYLINE_CASA").to_pylist()
    call_types = table.column("CALL_TYPE").to_pylist()
    
    for i in range(len(polylines)):
        _, _, end_lon, end_lat = extract_coords_from_polyline(polylines[i])
        if end_lon is None:
            continue
        
        # Pour simplifier, on utilise origin comme destination
        # À améliorer avec mapping lat/lon → zone
        dest = origin
        
        call_type = call_types[i]
        pair_counts[(origin, dest, call_type)] += 1
        call_counts[call_type] += 1
        total += 1
    
    print(f"   ✅ {table.num_rows} lignes traitées")

print(f"\n✅ {len(pair_counts)} paires uniques trouvées")
if total == 0:
    print("❌ Aucune donnée trouvée, utilisation de paires par défaut")
    weighted_pairs = [(1, 2, 'C', 0.5), (2, 3, 'B', 0.3), (3, 1, 'A', 0.2)]
else:
    weighted_pairs = [(o, d, c, cnt/total) for (o, d, c), cnt in pair_counts.items()]
call_weights = {c: cnt/total for c, cnt in call_counts.items()}
print(f"   CALL_TYPE : {call_weights}")

# ========== KAFKA PRODUCER ==========
producer = KafkaProducer(
    bootstrap_servers=KAFKA_BROKER,
    value_serializer=lambda v: json.dumps(v).encode("utf-8"),
    key_serializer=lambda k: str(k).encode("utf-8"),
    acks=1,
    retries=3,
    max_block_ms=60000,
    api_version=(3, 4, 0)
)
print("✅ Kafka connecté")

# ========== FONCTIONS ==========
def get_demand_weight():
    now = datetime.now()
    hour = now.hour
    w = HOURLY_WEIGHTS.get(hour, 0.5)
    if now.weekday() == 4 and hour in FRIDAY_OVERRIDE:
        w = FRIDAY_OVERRIDE[hour]
    if now.weekday() == 6:
        w *= 0.85
    return w

def choose_pair():
    r = random.random()
    cum = 0.0
    for o, d, c, prob in weighted_pairs:
        cum += prob
        if r <= cum:
            return o, d, c
    return weighted_pairs[-1][0], weighted_pairs[-1][1], weighted_pairs[-1][2]

# ========== BOUCLE PRINCIPALE ==========
print("\n📞 Trip Producer démarré")
count = 0
try:
    while True:
        weight = get_demand_weight()
        interval = (1.0 / max(weight, 0.1)) * (3 / SPEED_FACTOR)
        o, d, ct = choose_pair()
        event = {
            "trip_id": str(uuid.uuid4()),
            "rider_id": f"rider_{random.randint(1,50000):05d}",
            "origin_zone": o,
            "dest_zone": d,
            "requested_at": int(datetime.now().timestamp()),
            "call_type": ct,
            "demand_weight": round(weight, 2)
        }
        producer.send(TOPIC_TRIPS, key=event["trip_id"], value=event)
        count += 1
        if count % 10 == 0:
            print(f"   📍 {count} | heure={datetime.now().hour}h | poids={weight:.2f} | zone {o}→{d} ({ct})")
        time.sleep(interval)
except KeyboardInterrupt:
    print(f"\n⏹️ Arrêt — {count} réservations envoyées")
    producer.flush()
    producer.close()

📋 Liste des fichiers Parquet...
📁 2 fichiers trouvés
   - casa-trajectories/final_89688/part-00000-14480b7d-931b-441c-84fe-8a15cadedc10-c000.snappy.parquet
   - casa-trajectories/final_89688/part-00001-14480b7d-931b-441c-84fe-8a15cadedc10-c000.snappy.parquet

📖 Lecture de casa-trajectories/final_89688/part-00000-14480b7d-931b-441c-84fe-8a15cadedc10-c000.snappy.parquet
   Tentative 1/3...
   Colonnes: ['CALL_TYPE', 'DAY_TYPE', 'DOW', 'DURATION_S', 'HOUR', 'POLYLINE_CASA', 'TAXI_ID', 'TRIP_ID']
   ⚠️ origin_zone_id non trouvé, skip

📖 Lecture de casa-trajectories/final_89688/part-00001-14480b7d-931b-441c-84fe-8a15cadedc10-c000.snappy.parquet
   Tentative 1/3...
   Colonnes: ['CALL_TYPE', 'DAY_TYPE', 'DOW', 'DURATION_S', 'HOUR', 'POLYLINE_CASA', 'TAXI_ID', 'TRIP_ID']
   ⚠️ origin_zone_id non trouvé, skip

✅ 0 paires uniques trouvées
❌ Aucune donnée trouvée, utilisation de paires par défaut
   CALL_TYPE : {}
✅ Kafka connecté

📞 Trip Producer démarré
   📍 10 | heure=3h | poids=0.06 | zone 2

KeyboardInterrupt: 

In [ ]:
trip.terminate()

In [ ]:
!ps aux | grep python

In [ ]:
gps.kill()
trip.kill()